In [310]:
import json
import os
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from loguru import logger

from unga81.database import get_all_countries, get_db_connection
from unga81.config import EXTERNAL_DATA_DIR, PROCESSED_DATA_DIR
from unga81.restcountries import download_country_data

pd.options.display.max_columns = None
pd.options.display.max_rows = None
# pd.options.display.max_rows = 20

load_dotenv()

API_KEY = os.getenv("API_KEY_RESTCOUNTRIES")

In [311]:
conn = get_db_connection()
df = get_all_countries(conn=conn)
logger.info(f"{df.shape[0]} speeches in the database")

2026-09-28 04:08:56.808 | INFO     | __main__:<module>:3 - 176 speeches in the database


# RestcountriesAPI

In [ ]:
country_information_path = EXTERNAL_DATA_DIR / "restcountriesapi"
country_information_path.mkdir(parents=True, exist_ok=True)


for i, r in df.iterrows():
    if isinstance(r["iso_3"], str):
        download_country_data(alpha_3=r["iso_3"], output_path=country_information_path)

In [ ]:
# name_common = data_pais["data"]["objects"][0]["names"]["common"]
# name_official = data_pais["data"]["objects"][0]["names"]["official"]
# lat, lng = data_pais["data"]["objects"][0]["capitals"][0]["coordinates"]
# capital_name = data_pais["data"]["objects"][0]["capitals"][0]["name"]
alpha_3 = data_pais["data"]["objects"][0]["codes"]["alpha_3"]
# data_pais["data"]["objects"][0]["flag"]
alpha_3

In [ ]:
# TODO parse
# - flag
# - capital
# - area
# - timezones
# - currency
# - languages
# - borders
# - flag png
# - gini index
# - coat of arms

In [ ]:
pd.DataFrame(data_pais["data"]["objects"]).T

In [ ]:
def get_country_data(alpha_3: str):
    file_path = EXTERNAL_DATA_DIR / "restcountriesapi" / f"{alpha_3}.json"
    if not file_path.exists():
        return None

    with open(file_path, "r") as file:
        data = json.load(file)
    return data

In [ ]:
country_data = get_country_data("AGO")

In [ ]:
country_data["data"]["objects"][0]["flag"]["url_png"]
country_data["data"]["objects"][0]["area"]["kilometers"]
country_data["data"]["objects"][0]["borders"]
country_data["data"]["objects"][0]["economy"]["gini_coefficient"]
capitals = ", ".join(
    [x["name"] for x in country_data["data"]["objects"][0]["capitals"]]
)
country_data["data"]["objects"][0]["government_type"]
", ".join([x["name"] for x in country_data["data"]["objects"][0]["languages"]])
len(country_data["data"]["objects"][0]["languages"])

# Worldmeter

In [ ]:
def get_tables_from_website(url) -> list | None:
    response = requests.get(url)
    if response.ok:
        tables = pd.read_html(StringIO(response.text))
        logger.info(f"{len(tables)} tables found in {url}.")
        return tables
    return None


def parse_table(df: pd.DataFrame) -> pd.DataFrame: ...

In [ ]:
url_gdp_by_country = "https://www.worldometers.info/gdp/gdp-by-country/"
gdp_tables = get_tables_from_website(url_gdp_by_country)

In [ ]:
gdp_imf = gdp_tables[0]
gdp_wb = gdp_tables[1]

gdp_imf = gdp_imf[["Country", "GDP (Full Value, US$)", "GDP per Capita (US$)"]]
gdp_imf.columns = ["country", "gdp (us$) imf", "gdp per capita (us$) imf"]

gdp_wb = gdp_wb[["Country", "GDP (Full Value, US$)", "GDP per Capita (US$)"]]
gdp_wb.columns = ["country", "gdp (us$) wb", "gdp per capita (us$) wb"]

In [ ]:
df_gdp = gdp_imf.merge(gdp_wb, on="country")

In [ ]:
url_population = "https://www.worldometers.info/geography/countries-of-the-world/"
population_tables = get_tables_from_website(url=url_population)

In [ ]:
df_population = pd.concat([population_tables[0], population_tables[1]])
df_population = df_population[["Country", "Population 2026"]].reset_index(drop=True)
df_population.columns = ["country", "population 2026"]

In [ ]:
df_demographics = df_population.merge(df_gdp, on="country", how="left")

In [ ]:
# preprocessing data from the SDG report
country_map = {
    "CÃ´te d'Ivoire": "Côte d'Ivoire",
    "DR Congo": "Democratic Republic of the Congo",
    "Laos": "Lao People's Democratic Republic",
    "Saint Kitts & Nevis": "Saint Kitts and Nevis",
    "South Korea": "Republic of Korea",
    "Brunei": "Brunei Darussalam",
    "Czech Republic (Czechia)": "Czechia",
    "United States": "United States of America",
    "Vietnam": "Viet Nam",
    "Sao Tome & Principe": "Sao Tome and Principe",
    "St. Vincent & Grenadines": "Saint Vincent & Grenadines",
    "Russia": "Russian Federation",
}

for i, r in df_demographics.iterrows():
    if r["country"] in country_map:
        df_demographics.at[i, "country"] = country_map[r["country"]]

df[["country", "iso_3"]].merge(df_demographics, on="country", how="left").sort_values(
    by="country"
).to_csv(PROCESSED_DATA_DIR / "worldometer.csv", index=False)

df_aux = (
    df[["country", "iso_3"]]
    .merge(df_demographics, on="country", how="outer")
    .sort_values(by="country")
)

df_aux[(df_aux["iso_3"].isnull()) | (df_aux["population 2026"].isnull())]

# SDG Report

In [ ]:
df_sdg_data = pd.read_excel(
    EXTERNAL_DATA_DIR / "sdg_index" / "database_2026.xlsx", sheet_name="SDR2026 Data"
)

In [ ]:
df_sdg_data = df_sdg_data[
    [
        "Country Code ISO3",
        "country",
        "2026 SDG Index Score",
        "2026 SDG Index Rank",
        "Goal 1 Dash",
        "Goal 1 Trend",
        "Goal 2 Dash",
        "Goal 2 Trend",
        "Goal 3 Dash",
        "Goal 3 Trend",
        "Goal 4 Dash",
        "Goal 4 Trend",
        "Goal 5 Dash",
        "Goal 5 Trend",
        "Goal 6 Dash",
        "Goal 6 Trend",
        "Goal 7 Dash",
        "Goal 7 Trend",
        "Goal 8 Dash",
        "Goal 8 Trend",
        "Goal 9 Dash",
        "Goal 9 Trend",
        "Goal 10 Dash",
        "Goal 10 Trend",
        "Goal 11 Dash",
        "Goal 11 Trend",
        "Goal 12 Dash",
        "Goal 12 Trend",
        "Goal 13 Dash",
        "Goal 13 Trend",
        "Goal 14 Dash",
        "Goal 14 Trend",
        "Goal 15 Dash",
        "Goal 15 Trend",
        "Goal 16 Dash",
        "Goal 16 Trend",
        "Goal 17 Dash",
        "Goal 17 Trend",
        "Poverty headcount ratio at $3.00/day (%)",
    ]
]

In [ ]:
df[["country", "iso_3"]].merge(
    df_sdg_data, how="left", left_on="iso_3", right_on="Country Code ISO3"
).head(10)

In [ ]:
response = requests.get("https://dashboards.sdgindex.org/profiles/")
if response.ok:
    soup = BeautifulSoup(response.text, "html.parser")

In [ ]:
links = soup.find_all("a")
data = []
base_url = "https://dashboards.sdgindex.org/profiles/"
for link in links:
    if (
        link.get("href").startswith("/profiles/")
        and len(link.get("href").replace("/profiles/", "").replace("/", "").strip()) > 0
    ):
        country_name = link.text.strip()
        country_link = link.get("href").replace("/profiles/", "")
        data.append((country_name, base_url + country_link))
df_sdg_profiles = pd.DataFrame(data, columns=["country", "sdg_profile"])

In [ ]:
df_sdg = df_sdg_data.merge(df_sdg_profiles, on="country", how="left")

In [ ]:
# df_sdg.to_csv(PROCESSED_DATA_DIR / "sdg_countries.csv", index=False)

# Other names

In [ ]:
import curl_cffi
import pandas as pd
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_alternative_country_names"
response = curl_cffi.get(url, impersonate="chrome")
soup = BeautifulSoup(response.text, "html.parser")
tables = soup.find_all(class_="wikitable")

,code,name,alternative_names
0,,Abkhazia,Republic of Abkhazia|Apsny|Aphsny Axwynthkharr...
1,AFG,Afghanistan,Islamic Emirate of Afghanistan|Da Afġānistān I...
2,ALB,Albania,Republic of Albania|Republika e Shqipërisë|Arn...
3,DZA,Algeria,People's Democratic Republic of Algeria|al-Jaz...
4,AND,Andorra,Principality of Andorra|Principat d'Andorra|Pr...


In [ ]:
data = []
for table in tables:
    for row in table.find_all("tr")[1:]:  # ignores the column names row
        cell_data = row.find_all("td")
        if len(cell_data) == 3:  # only process when there are 3 columns
            code, name, names = cell_data
            country_name = name.find("a")["title"].strip()
            other_names = "|".join(
                [x.text.strip() for x in names.find_all("b") if len(x.text.strip()) > 2]
                + [
                    x.text.strip()
                    for x in names.find_all("i")
                    if len(x.text.strip()) > 2
                ]
            )
            data.append((code.text, country_name, other_names))

df_country_names = pd.DataFrame(data, columns=["code", "name", "alternative_names"])
df_country_names.head()

,code,name,alternative_names
0,,Abkhazia,Republic of Abkhazia|Apsny|Aphsny Axwynthkharr...
1,AFG,Afghanistan,Islamic Emirate of Afghanistan|Da Afġānistān I...
2,ALB,Albania,Republic of Albania|Republika e Shqipërisë|Arn...
3,DZA,Algeria,People's Democratic Republic of Algeria|al-Jaz...
4,AND,Andorra,Principality of Andorra|Principat d'Andorra|Pr...


In [ ]:
df_country_names[df_country_names["name"] == "Papua New Guinea"]
df_country_names[df_country_names["name"] == "Estonia"]["alternative_names"].values[
    0
].split("|")

['Republic of Estonia',
 'Esthonia',
 'Eesti',
 'Eesti Vabariik',
 'Viru',
 'Viro',
 'Estland',
 'Maarjamaa',
 'Igaunija']

In [334]:
for i, r in df_country_names.iterrows():
    for name in [x.lower().strip() for x in r["alternative_names"].split("|")]:
        if name in df[df["iso_3"] == "CHL"]["full_speech"].values[0].lower():
            print(r["name"], name)

Estonia viro
Iceland fold
Lithuania lita
Russia rus
South Africa south
United States america
United States usa
United States usa


In [321]:
df[df["iso_3"] == "CHL"]["full_speech"].values[0].lower().replace(".", "").replace(
    ",", ""
)

"a very very good afternoon to you all secretary general president of the general assembly heads of state and government distinguished delegates particular greetings to the delegation that has traveled with me from chile i wish to greet our minister for foreign affairs our minister for the environment for development and the family our ambassadors our senators and members of parliament and i wish to greet each and every chilean p person greetings and a warm embrace to you all 64 years ago in this same forum the then president of chile horgeandre described our country chile as a large narrow nation between the andes and the pacific extending from there till the moving expanse of the antarctic the land remains where it is what has changed is the world around it what has changed first and foremost is chile for the vast swades of the '9s and until the beginning of the 2000s chile was a success story and a model for many regions and nations our country implemented economic policies that pri